<a href="https://colab.research.google.com/github/xidoudou/ai-agents-in-langgraph/blob/main/Lesson_1_Student.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Lesson 1: Simple ReAct Agent from Scratch

In [ ]:
# based on https://til.simonwillison.net/llms/python-react-pattern

In [2]:
import openai
import re
import httpx
import os
from dotenv import load_dotenv

_ = load_dotenv()
from openai import OpenAI

In [8]:
from google.colab import userdata
os.environ["OPENAI_API_KEY"] = userdata.get("OPENAI_API_KEY")

In [9]:
client = OpenAI()

In [26]:
responses = client.responses.create(
    model="gpt-6-astra",
    input="Hello World"
)

In [27]:
print(responses)

Response(id='resp_033e00bd240fa127006ab11ce75b3887d2833744858385b22e', created_at=1789992167.0, error=None, incomplete_details=None, instructions=None, metadata={}, model='gpt-6-astra', object='response', output=[ResponseOutputMessage(id='msg_033e00bd240fa127006ab11ce8f2e087d2ac07e18079d3c96b', content=[ResponseOutputText(annotations=[], text='Hello, world! 👋', type='output_text', logprobs=[])], role='assistant', status='completed', type='message', phase='final_answer')], parallel_tool_calls=True, temperature=1.0, tool_choice='auto', tools=[], top_p=0.98, background=False, completed_at=1789992169.0, conversation=None, max_output_tokens=None, max_tool_calls=None, moderation=None, previous_response_id=None, prompt=None, prompt_cache_key=None, prompt_cache_options=None, prompt_cache_retention='24h', reasoning=Reasoning(context='all_turns', effort='medium', generate_summary=None, mode='standard', summary=None), safety_identifier=None, service_tier='default', status='completed', text=Respon

In [29]:
responses.output_text

'Hello, world! 👋'

In [31]:
class Agent:
    def __init__(self, system=""):
        self.system = system
        self.messages = []
        if self.system:
            self.messages.append({"role": "system", "content": system})

    def __call__(self, message):
        self.messages.append({"role": "user", "content": message})
        result = self.execute()
        self.messages.append({"role": "assistant", "content": result})
        return result

    def execute(self):
        responses = client.responses.create(
                        model="gpt-6-astra",
                        input =self.messages)
        return responses.output_text


In [32]:
prompt = """
You run in a loop of Thought, Action, PAUSE, Observation.
At the end of the loop you output an Answer
Use Thought to describe your thoughts about the question you have been asked.
Use Action to run one of the actions available to you - then return PAUSE.
Observation will be the result of running those actions.

Your available actions are:

calculate:
e.g. calculate: 4 * 7 / 3
Runs a calculation and returns the number - uses Python so be sure to use floating point syntax if necessary

average_dog_weight:
e.g. average_dog_weight: Collie
returns average weight of a dog when given the breed

Example session:

Question: How much does a Bulldog weigh?
Thought: I should look the dogs weight using average_dog_weight
Action: average_dog_weight: Bulldog
PAUSE

You will be called again with this:

Observation: A Bulldog weights 51 lbs

You then output:

Answer: A bulldog weights 51 lbs
""".strip()

In [39]:
def calculate(what):
    return eval(what)

def average_dog_weight(name):
    if name in "Scottish Terrier":
        return("Scottish Terriers average 20 lbs")
    elif name in "Border Collie":
        return("a Border Collies average weight is 37 lbs")
    elif name in "Toy Poodle":
        return("a toy poodles average weight is 7 lbs")
    else:
        return("An average dog weights 50 lbs")

known_actions = {
    "calculate": calculate,
    "average_dog_weight": average_dog_weight
}

In [40]:
abot = Agent(prompt)

In [41]:
result = abot("How much does a toy poodle weigh?")
print(result)

Thought: I’ll look up the average weight of a toy poodle.
Action: average_dog_weight: Toy Poodle
PAUSE pinning final?Thought: I’ll look up the average weight of a toy poodle.
Action: average_dog_weight: Toy Poodle
PAUSE


In [43]:
result = average_dog_weight("Toy Poodle")

In [44]:
result

'a toy poodles average weight is 7 lbs'

In [45]:
next_prompt = "Observation: {}".format(result)

In [46]:
abot(next_prompt)

'Answer: A toy poodle weighs about 7 lbs on average.'

In [47]:
abot.messages

[{'role': 'system',
  'content': 'You run in a loop of Thought, Action, PAUSE, Observation.\nAt the end of the loop you output an Answer\nUse Thought to describe your thoughts about the question you have been asked.\nUse Action to run one of the actions available to you - then return PAUSE.\nObservation will be the result of running those actions.\n\nYour available actions are:\n\ncalculate:\ne.g. calculate: 4 * 7 / 3\nRuns a calculation and returns the number - uses Python so be sure to use floating point syntax if necessary\n\naverage_dog_weight:\ne.g. average_dog_weight: Collie\nreturns average weight of a dog when given the breed\n\nExample session:\n\nQuestion: How much does a Bulldog weigh?\nThought: I should look the dogs weight using average_dog_weight\nAction: average_dog_weight: Bulldog\nPAUSE\n\nYou will be called again with this:\n\nObservation: A Bulldog weights 51 lbs\n\nYou then output:\n\nAnswer: A bulldog weights 51 lbs'},
 {'role': 'user', 'content': 'How much does a 

In [48]:
abot = Agent(prompt)

In [49]:
question = """I have 2 dogs, a border collie and a scottish terrier. \
What is their combined weight"""
abot(question)

'Thought: I’ll look up the average weight of each breed, then add them.\nAction: average_dog_weight: Border Collie\nPAUSE'

In [50]:
next_prompt = "Observation: {}".format(average_dog_weight("Border Collie"))
print(next_prompt)

Observation: a Border Collies average weight is 37 lbs


In [52]:
abot(next_prompt)

'Thought: That observation repeats the Border Collie’s weight. I still need the Scottish Terrier’s average weight.\nAction: average_dog_weight: Scottish Terrier\nPAUSE'

In [53]:
next_prompt = "Observation: {}".format(average_dog_weight("Scottish Terrier"))
print(next_prompt)

Observation: Scottish Terriers average 20 lbs


In [54]:
abot(next_prompt)

'Thought: I’ll add the two average weights.\nAction: calculate: 37 + 20\nPAUSE'

In [55]:
next_prompt = "Observation: {}".format(eval("37 + 20"))
print(next_prompt)

Observation: 57


In [56]:
abot(next_prompt)

'Answer: Based on breed averages, their combined weight is approximately **57 lbs** (37 lbs + 20 lbs).'

### Add loop

In [60]:
action_re = re.compile('^Action: (\w+): (.*)$')   # python regular expression to selection action

<>:1: SyntaxWarning: invalid escape sequence '\w'
<>:1: SyntaxWarning: invalid escape sequence '\w'
/tmp/ipykernel_5408/2389047760.py:1: SyntaxWarning: invalid escape sequence '\w'
  action_re = re.compile('^Action: (\w+): (.*)$')   # python regular expression to selection action


In [61]:
def query(question, max_turns=5):
    i = 0
    bot = Agent(prompt)
    next_prompt = question
    while i < max_turns:
        i += 1
        result = bot(next_prompt)
        print(result)
        actions = [
            action_re.match(a)
            for a in result.split('\n')
            if action_re.match(a)
        ]
        if actions:
            # There is an action to run
            action, action_input = actions[0].groups()
            if action not in known_actions:
                raise Exception("Unknown action: {}: {}".format(action, action_input))
            print(" -- running {} {}".format(action, action_input))
            observation = known_actions[action](action_input)
            print("Observation:", observation)
            next_prompt = "Observation: {}".format(observation)
        else:
            return

In [62]:
question = """I have 2 dogs, a border collie and a scottish terrier. \
What is their combined weight"""
query(question)

Thought: I’ll look up the average weight for each breed.
Action: average_dog_weight: Border Collie
PAUSE
 -- running average_dog_weight Border Collie
Observation: a Border Collies average weight is 37 lbs
Thought: I’ll look up the Scottish Terrier’s average weight next.
Action: average_dog_weight: Scottish Terrier
PAUSE
 -- running average_dog_weight Scottish Terrier
Observation: Scottish Terriers average 20 lbs
Thought: I’ll add the two breed averages.
Action: calculate: 37 + 20
PAUSE
 -- running calculate 37 + 20
Observation: 57
Answer: Based on breed averages, their combined weight is approximately **57 lbs** (37 lbs + 20 lbs).
